# Session 4 — From Model to Production: End-to-End ML

## 🎯 Session Objectives

1. **Understand the MLE lifecycle**: From training artifact → inference function → API → UI
2. **Model serialization**: Save checkpoints and reload them reproducibly at serving time
3. **Build a clean inference module** (`src/inference.py`): top-K scoring with ID encoding
4. **Deploy a REST API** (`app/main.py`): FastAPI endpoint serving live recommendations
5. **Build a simple demo** (`app/frontend.py`): Streamlit UI connected to your API
6. **Handle the cold-start problem**: Fallback strategies using the popularity baseline
7. **Synthesize all sessions**: Data → Model → Serving

---

## 🗺️ Session Map

| Part | Topic | Key File |
|------|-------|----------|
| **A** | System Architecture & the MLE Mindset | — conceptual — |
| **B** | Model Serialization | `data/models/ncf_checkpoint.pt` |
| **C** | Inference Pipeline | `src/inference.py` |
| **D** | REST API Backend | `app/main.py` |
| **E** | Streamlit Frontend | `app/frontend.py` |
| **F** | End-to-End Testing | — |
| **G** | Cold Start Problem | `src/inference.py` (extended) |
| **H** | Production Best Practices | — conceptual — |
| **I** | Future Directions | — |
| **J** | Final Project Report Checklist | — |

> **End Goal**: A user enters their user ID → clicks *Get Recommendations* → sees a ranked list of movies powered by your trained NCF model.

---

## 📁 Files You Will Complete This Session

```
src/
  inference.py     ← Core inference logic (load model, score items, return top-K)
app/
  main.py          ← FastAPI backend (startup model loading, /recommend endpoint)
  frontend.py      ← Streamlit UI (call API, display recommendations)
```

---
## 🏗️ PART A: SYSTEM ARCHITECTURE & THE MLE MINDSET

### A.1: Where Does the Data Scientist / MLE Fit?

A production recommendation system has several distinct layers. As a Data Scientist or MLE
you own the **ranking layer** and the **serving infrastructure** — taking the trained model
from a notebook to a live endpoint that products and downstream teams consume.

```
┌──────────────────┬──────────────────────────────────────────┐
│ 4. SERVICE LAYER │ REST API, caching, load balancing        │
│                  │ → FastAPI / Flask / TorchServe           │
│                  │ ← You implement this today               │
├──────────────────┼──────────────────────────────────────────┤
│ 3. RANKING LAYER │ Score candidates with your ML model      │
│                  │ → NCF / MF from Sessions 2–3             │
│                  │ ← You implement inference.py today       │
├──────────────────┼──────────────────────────────────────────┤
│ 2. RETRIEVAL     │ Narrow millions → hundreds of candidates │
│    LAYER         │ → ANN search, heuristic filters          │
│                  │ ← Simplified: score all items for demo   │
├──────────────────┼──────────────────────────────────────────┤
│ 1. DATA LAYER    │ User/item features, interaction logs     │
│                  │ → Ratings, embeddings, metadata          │
│                  │ ← Built in Sessions 1–3                  │
└──────────────────┴──────────────────────────────────────────┘
```

### A.2: Offline vs. Online Inference

| Mode | When computed | Typical use case |
|------|--------------|------------------|
| **Offline (Batch)** | Nightly job, pre-compute top-K for all users | Email campaigns, homepage |
| **Online (Real-time)** | Per user request, score items live | Search results, "You may also like" |

**This session** builds **online inference** (real-time API). The same `inference.py`
functions work for batch jobs with no changes.

### A.3: The Data Flow in One Inference Request

```
HTTP GET /recommend/42?k=10
        ↓
API Layer (app/main.py)
  → Validate input, retrieve model from memory
        ↓
Inference Layer (src/inference.py)
  → Encode user_id=42 → user_idx (model space)
  → Score all N items: model(user_idx, [0..N-1]) → score tensor
  → Top-K argmax → item indices → decode to item IDs
        ↓
HTTP Response: {"user_id": 42, "recommendations": [item_7, item_3, ...]}
```

> 🔑 **Key insight**: The model outputs a score ∈ [0,1] per (user, item) pair.
> Recommendations are simply the **top-K items sorted by that score**.

In [ ]:
## ============================================================================
## PART A / SETUP: IMPORTS & PATHS
## ============================================================================

import sys
import os
import json
import time
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn

try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False
    print('⚠️  Install requests: pip install requests')

# Add project root to path so we can import src.*
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_DIR = project_root / 'data' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'✅ Project root : {project_root}')
print(f'✅ Device       : {DEVICE}')
print(f'✅ PyTorch      : {torch.__version__}')
print(f'✅ Model dir    : {MODEL_DIR}')

In [ ]:
## ============================================================================
## LOAD DATA  (consistent with Sessions 1–3)
## ============================================================================

DATA_PATH = project_root / 'data' / 'processed' / 'ratings.csv'

try:
    ratings_df = pd.read_csv(str(DATA_PATH))
    print(f'✅ Loaded {len(ratings_df):,} ratings from {DATA_PATH.name}')
except FileNotFoundError:
    print('⚠️  ratings.csv not found — generating synthetic data for demo...')
    np.random.seed(42)
    n_users, n_items, n_ratings = 500, 200, 8000
    ratings_df = pd.DataFrame({
        'user_id': np.random.randint(0, n_users, n_ratings),
        'item_id': np.random.randint(0, n_items, n_ratings),
        'rating':  np.random.choice([1, 2, 3, 4, 5], n_ratings),
    })
    ratings_df = (
        ratings_df
        .sort_values('rating', ascending=False)
        .drop_duplicates(subset=['user_id', 'item_id'], keep='first')
        .reset_index(drop=True)
    )

# Build contiguous ID encoders (required by embedding layers)
user_ids   = sorted(ratings_df['user_id'].unique())
item_ids   = sorted(ratings_df['item_id'].unique())
user2idx   = {u: i for i, u in enumerate(user_ids)}
idx2user   = {i: u for u, i in user2idx.items()}
item2idx   = {it: i for i, it in enumerate(item_ids)}
idx2item   = {i: it for it, i in item2idx.items()}

NUM_USERS = len(user2idx)
NUM_ITEMS = len(item2idx)

print(f'\n📊 Dataset')
print(f'  Users   : {NUM_USERS}')
print(f'  Items   : {NUM_ITEMS}')
print(f'  Ratings : {len(ratings_df):,}')
sparsity = 1 - len(ratings_df) / (NUM_USERS * NUM_ITEMS)
print(f'  Sparsity: {sparsity:.2%}')
print(f'\n  Sample:\n{ratings_df.head(3)}')

---
## 💾 PART B: MODEL SERIALIZATION — Save & Load Checkpoints

### B.1: Why Checkpoints Matter for MLEs

A trained model lives in GPU/CPU memory during training. To **serve it later** (in an API,
a batch job, or a different machine) you must:

1. **Save** the model weights *and* the architecture hyperparameters to disk.
2. **Load** them back at serving time — without needing access to the original training script.

**Best practice — save a checkpoint dict:**

```python
# ✅ GOOD: Save state dict + all params needed to reconstruct the architecture
checkpoint = {
    'model_state_dict': model.state_dict(),
    'num_users': num_users,
    'num_items': num_items,
    'embedding_dim': 32,
    'mlp_hidden_dims': [128, 64, 32],
    'user2idx': user2idx,   # ID encoder — required at serving time
    'item2idx': item2idx,
}
torch.save(checkpoint, 'model.pt')

# ❌ AVOID: Pickling the whole model object (breaks when code changes)
torch.save(model, 'model.pt')
```

### B.2: Key Point — ID Encoders Must Travel With the Model

The raw `user_id` / `item_id` values in the database (e.g. user_id=42) are **not** the
embedding row indices. The encoder maps `raw_id → contiguous index`.
If you save the model but lose the encoder, you cannot serve predictions correctly.

In [ ]:
## ============================================================================
## PART B.1: SAVE A MODEL CHECKPOINT
## Run this cell to create a checkpoint for the rest of the session.
## If you completed Session 3, replace the model weights below with your
## trained NCF weights.
## ============================================================================

from src.models.neural_cf import NeuralCF

EMBEDDING_DIM  = 32
MLP_HIDDEN     = [128, 64, 32]
CHECKPOINT_PATH = MODEL_DIR / 'ncf_checkpoint.pt'

# ── Instantiate the model (random weights if not trained yet) ────────────────
model = NeuralCF(
    num_users=NUM_USERS,
    num_items=NUM_ITEMS,
    embedding_dim=EMBEDDING_DIM,
    mlp_hidden_dims=MLP_HIDDEN,
    dropout_rate=0.3,
)

# Optional: load trained weights from Session 3
# trained_path = MODEL_DIR / 'ncf_trained.pt'
# if trained_path.exists():
#     model.load_state_dict(torch.load(trained_path, map_location='cpu'))
#     print('✅ Loaded trained weights from Session 3')

# ── Pack checkpoint dict ─────────────────────────────────────────────────────
checkpoint = {
    'model_state_dict': model.state_dict(),
    'num_users':        NUM_USERS,
    'num_items':        NUM_ITEMS,
    'embedding_dim':    EMBEDDING_DIM,
    'mlp_hidden_dims':  MLP_HIDDEN,
    'dropout_rate':     0.3,
    # ID encoders — must be saved alongside weights
    'user2idx':         user2idx,
    'item2idx':         item2idx,
}

torch.save(checkpoint, str(CHECKPOINT_PATH))
print(f'✅ Checkpoint saved → {CHECKPOINT_PATH}')
print(f'   File size : {CHECKPOINT_PATH.stat().st_size / 1024:.1f} KB')
print(f'   Keys      : {list(checkpoint.keys())}')

In [ ]:
## ============================================================================
## PART B.2 — TODO: Load the Checkpoint
##
## Implement load_checkpoint() below. This pattern is the foundation
## of the load_model() function you will write in src/inference.py.
##
## Steps:
##   1. Call torch.load(path, map_location=device) → returns the checkpoint dict
##   2. Reconstruct NeuralCF using the saved hyperparameters:
##      num_users, num_items, embedding_dim, mlp_hidden_dims, dropout_rate
##   3. Call model.load_state_dict(checkpoint['model_state_dict'])
##   4. Call model.eval()  ← disables dropout during inference
##   5. Return (model, checkpoint)
## ============================================================================

def load_checkpoint(checkpoint_path: str, device: str = 'cpu'):
    """
    Load an NCF model from a checkpoint file.

    Args:
        checkpoint_path: path to the .pt checkpoint file
        device: 'cpu' or 'cuda'

    Returns:
        (model, checkpoint_dict)
    """
    # YOUR CODE HERE
    pass


# ── Validate ────────────────────────────────────────────────────────────────
result = load_checkpoint(str(CHECKPOINT_PATH))

if result is not None:
    model_loaded, ckpt_loaded = result
    print('✅ Model loaded successfully!')
    print(f'   Training mode : {model_loaded.training}')   # should be False
    print(f'   Parameters    : {sum(p.numel() for p in model_loaded.parameters()):,}')
    print(f'   user2idx keys : {len(ckpt_loaded["user2idx"])} users')
else:
    print('❌ load_checkpoint() returned None — implement it above!')
    model_loaded = model   # fallback so later cells still run
    ckpt_loaded  = checkpoint

---
## 🔧 PART C: INFERENCE PIPELINE — `src/inference.py`

### C.1: What the Inference Module Must Do

The inference module is the **bridge between your trained model and the API**. It must:

| Step | Operation |
|------|-----------|
| 1 | Accept a raw `user_id` (from the database / HTTP request) |
| 2 | Encode it to a model index using `user2idx` |
| 3 | Score **all items** in one vectorized forward pass |
| 4 | Decode the top-K indices back to original `item_id` values |
| 5 | Return an ordered list of recommended item IDs |

### C.2: Vectorized Batch Scoring (efficient)

Instead of calling `model(user, item)` once per item, we score all N items in a **single
forward pass** by repeating the user index:

```python
num_items   = model.num_items
user_tensor = torch.tensor([user_idx] * num_items)   # shape: (N_items,)
item_tensor = torch.tensor(list(range(num_items)))    # shape: (N_items,)

with torch.no_grad():
    scores = model(user_tensor, item_tensor).squeeze()  # shape: (N_items,)

top_k_indices = scores.topk(k).indices.tolist()
top_k_items   = [idx2item[i] for i in top_k_indices]
```

> 💡 For catalogs with millions of items you would use **Approximate Nearest Neighbor
> (ANN)** search (FAISS, ScaNN) instead. For this project, exhaustive scoring is fine.

### C.3: Functions to Implement in `src/inference.py`

| Function | Signature | Description |
|----------|-----------|-------------|
| `load_model` | `(checkpoint_path, device) → (model, ckpt)` | Load checkpoint, return model + encoders |
| `recommend` | `(model, user_id, k, user2idx, idx2item, device) → List[int]` | Top-K for a known user |
| `recommend_with_fallback` | `(..., popular_items) → dict` | Handles unknown users (Part G) |

In [ ]:
## ============================================================================
## PART C.1 — TODO: Implement load_model() in src/inference.py
##
## Open: src/inference.py
## Add the following imports at the top:
##
##   import torch
##   from typing import Any, Dict, List, Optional, Tuple
##   from src.models.neural_cf import NeuralCF
##
## Then replace the placeholder recommend() with the full implementation:
##
##   def load_model(checkpoint_path: str, device: str = 'cpu'):
##       """
##       Steps:
##       1. Load checkpoint dict:
##              ckpt = torch.load(checkpoint_path, map_location=device)
##       2. Instantiate NeuralCF with saved hyperparameters:
##              model = NeuralCF(
##                  num_users=ckpt['num_users'],
##                  num_items=ckpt['num_items'],
##                  embedding_dim=ckpt['embedding_dim'],
##                  mlp_hidden_dims=ckpt['mlp_hidden_dims'],
##                  dropout_rate=ckpt['dropout_rate'],
##              )
##       3. Load weights: model.load_state_dict(ckpt['model_state_dict'])
##       4. Move to device and set eval: model.to(device).eval()
##       5. Return (model, ckpt)
##       """
##       pass  # YOUR CODE HERE
##
## ── Prototype here — copy to src/inference.py once it works ─────────────────
## ============================================================================

def load_model_proto(checkpoint_path: str, device: str = 'cpu'):
    """Prototype — copy to src/inference.py as load_model() when working."""
    # YOUR CODE HERE
    pass


# ── Smoke test ───────────────────────────────────────────────────────────────
res = load_model_proto(str(CHECKPOINT_PATH))
if res is not None:
    m_proto, c_proto = res
    print(f'✅ load_model_proto works — model.training = {m_proto.training}')
else:
    print('❌ Implement load_model_proto() above, then move to src/inference.py')

In [ ]:
## ============================================================================
## PART C.2 — TODO: Implement recommend() in src/inference.py
##
## Open: src/inference.py
## Replace the placeholder recommend() stub with:
##
##   def recommend(
##       model,
##       user_id: int,
##       k: int = 10,
##       user2idx: dict = None,
##       idx2item: dict = None,
##       device: str = 'cpu',
##   ) -> List[int]:
##       """
##       Steps:
##       1. Validate user_id is known:
##              if user_id not in user2idx:
##                  raise ValueError(f'Unknown user_id: {user_id}')
##       2. Encode: user_idx = user2idx[user_id]
##       3. Get item count: num_items = model.num_items
##       4. Build tensors and move to device:
##              user_t = torch.tensor([user_idx] * num_items).to(device)
##              item_t = torch.tensor(list(range(num_items))).to(device)
##       5. Forward pass (no gradients):
##              with torch.no_grad():
##                  scores = model(user_t, item_t).squeeze()
##       6. Get top-K indices: top_k = scores.topk(k).indices.tolist()
##       7. Decode and return: return [idx2item[i] for i in top_k]
##       """
##       pass  # YOUR CODE HERE
##
## ── Prototype here ───────────────────────────────────────────────────────────
## ============================================================================

def recommend_proto(
    model,
    user_id: int,
    k: int = 10,
    user2idx: dict = None,
    idx2item: dict = None,
    device: str = 'cpu',
):
    """Prototype — copy to src/inference.py as recommend() when working."""
    # YOUR CODE HERE
    pass


# ── Test with a known user ────────────────────────────────────────────────────
test_user = user_ids[0]
recs = recommend_proto(
    model_loaded, test_user, k=10,
    user2idx=user2idx, idx2item=idx2item,
)

if recs:
    print(f'✅ Top-10 for user {test_user}: {recs}')
    print(f'   All item IDs valid: {all(r in item2idx for r in recs)}')
else:
    print('❌ Implement recommend_proto() above!')

# ── Test error handling for unknown user ────────────────────────────────────
try:
    recommend_proto(model_loaded, 99999999, k=5, user2idx=user2idx, idx2item=idx2item)
    print('⚠️  Should have raised ValueError for unknown user!')
except (ValueError, TypeError):
    print('✅ Correctly raises error for unknown user_id')
except Exception as e:
    print(f'⚠️  Got unexpected error type: {type(e).__name__}: {e}')

In [ ]:
## ============================================================================
## PART C.3: TEST YOUR INFERENCE MODULE (import from src/inference.py)
##
## After you have filled in src/inference.py, run this cell to confirm
## the module imports and works end-to-end.
## ============================================================================

import importlib

try:
    import src.inference as inf_module
    importlib.reload(inf_module)   # pick up latest edits without restarting kernel

    m_srv, c_srv = inf_module.load_model(str(CHECKPOINT_PATH))
    u2i_srv  = c_srv['user2idx']
    i2it_srv = {i: it for it, i in c_srv['item2idx'].items()}

    recs_srv = inf_module.recommend(
        m_srv, user_ids[0], k=10,
        user2idx=u2i_srv, idx2item=i2it_srv,
    )
    print(f'✅ src.inference.recommend() works — {len(recs_srv)} items returned')
    print(f'   First 5: {recs_srv[:5]}')

except AttributeError as e:
    print(f'❌ Function missing in src/inference.py: {e}')
    print('   Make sure load_model() and recommend() are implemented and saved.')
except Exception as e:
    print(f'❌ Error: {type(e).__name__}: {e}')

In [ ]:
## ============================================================================
## PART C.4: VISUALIZE ITEM SCORE DISTRIBUTIONS
##
## This cell shows what the model's raw output looks like before top-K.
## Run after Part C.2 works.
## ============================================================================

def score_all_items(model, user_idx: int, device: str = 'cpu'):
    """Return raw sigmoid scores for every item (for visualization)."""
    n = model.num_items
    u_t = torch.tensor([user_idx] * n).to(device)
    i_t = torch.tensor(list(range(n))).to(device)
    model.eval()
    with torch.no_grad():
        scores = model(u_t, i_t).squeeze().cpu().numpy()
    return scores


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: score distributions for 3 sample users
sample_user_idxs = [user2idx[u] for u in user_ids[:3]]
for uid_idx in sample_user_idxs:
    sc = score_all_items(model_loaded, uid_idx)
    axes[0].hist(sc, bins=30, alpha=0.55, label=f'User idx {uid_idx}')
axes[0].set_title('Score Distribution per User', fontsize=13)
axes[0].set_xlabel('Predicted Score (sigmoid output)')
axes[0].set_ylabel('Item Count')
axes[0].legend()

# Right: bar chart of top-10 items for the first user
sc0 = score_all_items(model_loaded, sample_user_idxs[0])
top10_idx = np.argsort(sc0)[::-1][:10]
axes[1].barh(
    range(10),
    sc0[top10_idx[::-1]],
    color='steelblue',
)
axes[1].set_yticks(range(10))
axes[1].set_yticklabels([f'Item {idx2item[i]}' for i in top10_idx[::-1]])
axes[1].set_title(f'Top-10 Items for User {idx2user[sample_user_idxs[0]]}', fontsize=13)
axes[1].set_xlabel('Predicted Score')

plt.suptitle('NCF Model Score Visualization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ Score visualization complete.')

---
## 🚀 PART D: REST API BACKEND — `app/main.py`

### D.1: Why FastAPI?

FastAPI is the standard ML-serving framework in Python:
- **Auto-generated docs**: visit `http://localhost:8000/docs` — no extra work
- **Pydantic validation**: request/response schemas enforced automatically
- **Async**: handles concurrent requests without blocking
- **Fast**: uvicorn + Starlette, production-grade throughput

### D.2: Load the Model Once at Startup — Never Per Request

| Strategy | Latency | Correct? |
|----------|---------|----------|
| Load model at startup | ~0 ms per call | ✅ |
| Load model per request | 1–10 s per call | ❌ |

FastAPI's `lifespan` context manager is the modern way to run startup/shutdown logic:

```python
from contextlib import asynccontextmanager

_state = {}   # shared model state

@asynccontextmanager
async def lifespan(app):
    _state['model'], _state['ckpt'] = load_model('data/models/ncf_checkpoint.pt')
    yield         # ← API runs here
    _state.clear()

app = FastAPI(lifespan=lifespan)
```

### D.3: Endpoints to Implement

```
GET  /                        → Health check
GET  /model-info              → num_users, num_items, embedding_dim
GET  /recommend/{user_id}     → Top-K item IDs  (query param: k, default=10)
```

In [ ]:
## ============================================================================
## PART D.1 — TODO: Complete app/main.py
##
## Open: app/main.py   (already has a skeleton — fill in the TODOs)
##
## ─── IMPORTS ────────────────────────────────────────────────────────────────
##   from contextlib import asynccontextmanager
##   from fastapi import FastAPI, HTTPException
##   from pydantic import BaseModel
##   from typing import List
##   import torch
##   from pathlib import Path
##   from src.inference import load_model, recommend, recommend_with_fallback
##
## ─── GLOBAL STATE ───────────────────────────────────────────────────────────
##   _state = {}   # holds model, encoders, metadata
##
## ─── LIFESPAN ───────────────────────────────────────────────────────────────
##   @asynccontextmanager
##   async def lifespan(app):
##       CKPT = Path('data/models/ncf_checkpoint.pt')
##       model, ckpt = load_model(str(CKPT))          # Step 1: load model
##       _state['model']    = model                   # Step 2: store in state
##       _state['user2idx'] = ckpt['user2idx']
##       _state['idx2item'] = {i: it for it, i in ckpt['item2idx'].items()}
##       _state['metadata'] = {
##           'num_users':     ckpt['num_users'],
##           'num_items':     ckpt['num_items'],
##           'embedding_dim': ckpt['embedding_dim'],
##       }
##       # Step 3: build popularity fallback (for cold-start, added in Part G)
##       # _state['popular_items'] = ...
##       yield
##       _state.clear()
##
## ─── APP ────────────────────────────────────────────────────────────────────
##   app = FastAPI(title='Movie Recommendation API', version='1.0',
##                 lifespan=lifespan)
##
## ─── RESPONSE SCHEMA ────────────────────────────────────────────────────────
##   class RecommendationResponse(BaseModel):
##       user_id: int
##       recommendations: List[int]
##       k: int
##       strategy: str = 'personalized'
##
## ─── ENDPOINTS ──────────────────────────────────────────────────────────────
##   @app.get('/')
##   async def health_check():
##       return {'status': 'ok'}
##
##   @app.get('/model-info')
##   async def model_info():
##       # TODO: return _state['metadata']
##       pass
##
##   @app.get('/recommend/{user_id}', response_model=RecommendationResponse)
##   async def recommend_endpoint(user_id: int, k: int = 10):
##       # Step 1: validate k (1 <= k <= 50), raise HTTPException(400) if invalid
##       # Step 2: get model from _state, raise HTTPException(503) if not ready
##       # Step 3: call recommend_with_fallback(...) from src/inference.py
##       # Step 4: return RecommendationResponse(...)
##       pass
##
## ─── START ──────────────────────────────────────────────────────────────────
##   Run with: uvicorn app.main:app --reload
##   Docs at : http://localhost:8000/docs
## ============================================================================

print('📋 TODO: Open app/main.py and implement the structure described above.')
print('   Once done, start the server and proceed to Part D.2 to test it.')

## ── Offline simulation (tests your logic without starting a server) ──────────
print('\n── Offline Simulation ──────────────────────────────────────────────────')

def _simulate_endpoint(user_id: int, k: int = 10):
    """Replicates the /recommend endpoint logic for offline testing."""
    if not (1 <= k <= 50):
        return {'status': 400, 'detail': f'k must be 1-50, got {k}'}
    if user_id not in user2idx:
        return {'status': 404, 'detail': f'User {user_id} not found'}

    uid_idx = user2idx[user_id]
    n = model_loaded.num_items
    u_t = torch.tensor([uid_idx] * n)
    i_t = torch.tensor(list(range(n)))
    model_loaded.eval()
    with torch.no_grad():
        scores = model_loaded(u_t, i_t).squeeze()
    top_k = scores.topk(k).indices.tolist()
    recs  = [idx2item[i] for i in top_k]
    return {'status': 200, 'user_id': user_id, 'recommendations': recs, 'k': k}

result = _simulate_endpoint(user_ids[0], k=5)
print(f'✅ Simulated /recommend/{user_ids[0]}?k=5 → HTTP {result["status"]}')
print(f'   Recommendations: {result.get("recommendations", [])}')

bad = _simulate_endpoint(99999999, k=5)
print(f'✅ Simulated /recommend/99999999 → HTTP {bad["status"]} ({bad["detail"]})')

In [ ]:
## ============================================================================
## PART D.2: LIVE API TESTS
##
## Start the server in a terminal first:
##   uvicorn app.main:app --reload
##
## Then run this cell to validate every endpoint.
## ============================================================================

API_BASE    = 'http://localhost:8000'
TEST_USER   = user_ids[0]

if not REQUESTS_AVAILABLE:
    print('⚠️  pip install requests')
else:
    cases = [
        ('Health check',        'GET', f'{API_BASE}/',                           200),
        ('Model info',          'GET', f'{API_BASE}/model-info',                 200),
        (f'Known user k=5',     'GET', f'{API_BASE}/recommend/{TEST_USER}?k=5',  200),
        ('Unknown user',        'GET', f'{API_BASE}/recommend/99999999',          404),
        ('Invalid k=0',         'GET', f'{API_BASE}/recommend/{TEST_USER}?k=0',  400),
        ('Invalid k=100',       'GET', f'{API_BASE}/recommend/{TEST_USER}?k=100',400),
    ]

    passed = 0
    for name, method, url, expected in cases:
        try:
            r = requests.request(method, url, timeout=5)
            ok = r.status_code == expected
            icon = '✅' if ok else '❌'
            print(f'  {icon} {name:<25} → HTTP {r.status_code} (expected {expected})')
            if ok:
                passed += 1
        except requests.ConnectionError:
            print(f'  ❌ {name:<25} → Connection refused — is the server running?')
        except Exception as e:
            print(f'  ❌ {name:<25} → {e}')

    print(f'\n  Result: {passed}/{len(cases)} tests passed')

---
## 🖥️ PART E: STREAMLIT FRONTEND — `app/frontend.py`

### E.1: What Is Streamlit?

Streamlit lets data scientists build **interactive web UIs in pure Python** — no HTML, CSS,
or JavaScript required. It is the standard tool for ML demos and internal dashboards.

```python
import streamlit as st
st.title('My App')
user_id = st.number_input('User ID', value=1)
if st.button('Submit'):
    st.write(f'You entered {user_id}')
```
That's it — a running web app.

### E.2: The Request Flow

```
Browser (http://localhost:8501)
     ↓   user enters ID, clicks button
Streamlit (app/frontend.py)
     ↓   requests.get('/recommend/{user_id}')
FastAPI (app/main.py at http://localhost:8000)
     ↓   calls src/inference.py
NCF Model → scores → top-K
     ↑   JSON response
Streamlit → renders table / cards
```

### E.3: What Your Frontend Should Do

| Feature | Streamlit widget |
|---------|------------------|
| User ID input | `st.number_input()` |
| Number of recs slider | `st.slider()` |
| Submit button | `st.button()` |
| Results table | `st.dataframe()` or `st.table()` |
| API status in sidebar | `st.sidebar` + `requests.get('/')` |
| Error messages | `st.error()` / `st.warning()` |

In [ ]:
## ============================================================================
## PART E.1 — TODO: Complete app/frontend.py
##
## Open: app/frontend.py
##
## Implement the following structure:
##
## ─── IMPORTS ────────────────────────────────────────────────────────────────
##   import streamlit as st
##   import requests
##   import pandas as pd
##
## ─── PAGE CONFIG ────────────────────────────────────────────────────────────
##   st.set_page_config(page_title='Movie Recommender', page_icon='🎬',
##                      layout='centered')
##   st.title('🎬 Movie Recommendation Demo')
##   st.markdown('Enter a user ID to get personalized movie recommendations.')
##
## ─── SIDEBAR: API HEALTH ─────────────────────────────────────────────────────
##   with st.sidebar:
##       st.header('⚙️ Settings')
##       API_BASE = st.text_input('API Base URL', value='http://localhost:8000')
##       st.markdown('**API Status**')
##       # TODO: ping GET /  and show ✅ Connected or ❌ Offline
##       try:
##           r = requests.get(f'{API_BASE}/', timeout=2)
##           if r.status_code == 200:
##               st.success('✅ Connected')
##           else:
##               st.error(f'❌ HTTP {r.status_code}')
##       except Exception:
##           st.error('❌ Offline — start the backend server')
##
## ─── MAIN FORM ──────────────────────────────────────────────────────────────
##   col1, col2 = st.columns([2, 1])
##   with col1:
##       user_id = st.number_input('User ID', min_value=0, value=1, step=1)
##   with col2:
##       k = st.slider('# Recommendations', min_value=1, max_value=20, value=10)
##
##   if st.button('🔍 Get Recommendations', use_container_width=True):
##       with st.spinner('Fetching recommendations...'):
##           try:
##               r = requests.get(f'{API_BASE}/recommend/{user_id}',
##                                params={'k': k}, timeout=10)
##               # TODO: handle 200 → show table
##               # TODO: handle 404 → show warning (cold-start)
##               # TODO: handle other errors → show error
##           except requests.ConnectionError:
##               st.error('Cannot connect to API. Start the backend first.')
##
## ─── DISPLAY RESULTS (200 OK) ───────────────────────────────────────────────
##   if r.status_code == 200:
##       data = r.json()
##       st.success(f'Top {k} recommendations for User {user_id} '
##                  f'(strategy: {data.get("strategy", "personalized")})')
##       df = pd.DataFrame({'Rank': range(1, k+1),
##                          'Item ID': data['recommendations']})
##       st.dataframe(df, use_container_width=True)
##
## ─── RUN ────────────────────────────────────────────────────────────────────
##   streamlit run app/frontend.py
## ============================================================================

print('📋 TODO: Open app/frontend.py and implement the Streamlit UI above.')
print('   Run with: streamlit run app/frontend.py')

---
## ▶️ PART F: RUNNING THE FULL STACK

### F.1: Launch Order (two terminals)

```bash
# Terminal 1 — FastAPI backend
uvicorn app.main:app --reload
# API:  http://localhost:8000
# Docs: http://localhost:8000/docs

# Terminal 2 — Streamlit frontend
streamlit run app/frontend.py
# UI:   http://localhost:8501
```

### F.2: Manual Smoke Test Checklist

| # | Check | Command | Expected |
|---|-------|---------|----------|
| 1 | API health | `curl http://localhost:8000/` | `{"status":"ok"}` |
| 2 | Model info | `curl http://localhost:8000/model-info` | num_users, num_items |
| 3 | Known user | `curl http://localhost:8000/recommend/1?k=5` | 5 item IDs |
| 4 | Unknown user | `curl http://localhost:8000/recommend/99999` | popular fallback or 404 |
| 5 | Bad k | `curl "http://localhost:8000/recommend/1?k=0"` | HTTP 400 |
| 6 | UI loads | `http://localhost:8501` | Form displayed |
| 7 | UI result | Enter a user ID, click button | Recommendations table |

In [ ]:
## ============================================================================
## PART F.1: AUTOMATED SMOKE TEST SUITE
## Run with the FastAPI server already started.
## ============================================================================

def run_smoke_tests(api_base: str = 'http://localhost:8000', test_user_id: int = 1):
    """Full smoke test suite — call after starting uvicorn."""
    if not REQUESTS_AVAILABLE:
        print('⚠️  pip install requests')
        return

    cases = [
        ('Health check',           f'{api_base}/',                               200),
        ('Model info',             f'{api_base}/model-info',                     200),
        (f'Known user (k=5)',      f'{api_base}/recommend/{test_user_id}?k=5',   200),
        ('Unknown user',           f'{api_base}/recommend/99999999',             [200, 404]),
        ('Invalid k=0',            f'{api_base}/recommend/{test_user_id}?k=0',   400),
        ('Invalid k=100',          f'{api_base}/recommend/{test_user_id}?k=100', 400),
    ]

    passed = 0
    for name, url, expected in cases:
        expected_list = expected if isinstance(expected, list) else [expected]
        try:
            r = requests.get(url, timeout=5)
            ok   = r.status_code in expected_list
            icon = '✅' if ok else '❌'
            print(f'  {icon} {name:<28} → HTTP {r.status_code}')
            if ok:
                passed += 1
        except requests.ConnectionError:
            print(f'  ❌ {name:<28} → Connection refused')
        except Exception as e:
            print(f'  ❌ {name:<28} → {e}')

    total = len(cases)
    icon  = '✅' if passed == total else '⚠️ '
    print(f'\n  {icon} {passed}/{total} tests passed')


print('🔬 Running smoke tests against http://localhost:8000')
print('   (Start the server first: uvicorn app.main:app --reload)\n')
run_smoke_tests(test_user_id=user_ids[0])

---
## 🥶 PART G: THE COLD-START PROBLEM — A Core Data Science Challenge

### G.1: What Is Cold Start?

A **new user** has no interaction history. The model has no embedding for them:

```
Request: GET /recommend/user_id=99999
Inference: user2idx.get(99999) → None → KeyError / ValueError ❌
```

This is one of the most important open problems in recommender systems, and handling it
gracefully is what separates a production-ready MLE from a notebook-only data scientist.

### G.2: Fallback Strategies

| Strategy | Description | Effort | Quality |
|----------|-------------|--------|---------|
| **Popularity fallback** | Return globally most-interacted items | Low | Baseline |
| **Demographics** | Match to similar known users by age/gender | Medium | Better |
| **Onboarding survey** | Ask user to rate a few items at signup | Medium | Good |
| **Content-based** | Embed item metadata (genre, tags) — no history needed | Medium-High | Good |
| **Session-based** | Recommend items similar to what user just viewed (anonymous) | High | Very good |

**For this project**: implement the **popularity fallback** as a minimum viable solution.

### G.3: What to Add in `src/inference.py`

```python
def recommend_with_fallback(
    model, user_id, k=10,
    user2idx=None, idx2item=None,
    popular_items=None, device='cpu',
) -> dict:
    """
    Returns {"items": [...], "strategy": "personalized" | "popularity_fallback"}
    """
```

In [ ]:
## ============================================================================
## PART G.1: BUILD THE POPULARITY FALLBACK
## ============================================================================

from src.models.popularity import PopularityRecommender

popularity_model = PopularityRecommender(top_k=50)
popularity_model.fit(ratings_df, item_col='item_id')
popular_items = popularity_model.popular_items   # sorted by interaction count

print(f'✅ Popularity model built — {len(popular_items)} items ranked')
print(f'   Top-10 most popular items: {popular_items[:10]}')

# ── Visualise the popularity distribution ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: top-15 bar chart
top15     = popular_items[:15]
top15_cnt = [ratings_df[ratings_df['item_id'] == it].shape[0] for it in top15]
axes[0].barh(range(len(top15)), top15_cnt[::-1], color='salmon')
axes[0].set_yticks(range(len(top15)))
axes[0].set_yticklabels([f'Item {it}' for it in top15[::-1]])
axes[0].set_title('Top-15 Items by Interaction Count', fontsize=12)
axes[0].set_xlabel('Number of Interactions')

# Right: overall item popularity distribution (long tail)
item_counts = ratings_df['item_id'].value_counts().values
axes[1].hist(item_counts, bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('Long-Tail Item Popularity Distribution', fontsize=12)
axes[1].set_xlabel('Interactions per Item')
axes[1].set_ylabel('Number of Items')

plt.suptitle('Popularity Analysis — Cold Start Fallback Pool', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ Notice the long tail — most items have very few interactions.')

In [ ]:
## ============================================================================
## PART G.2 — TODO: Implement recommend_with_fallback() in src/inference.py
##
## Open: src/inference.py
## Add the following function AFTER recommend():
##
##   def recommend_with_fallback(
##       model,
##       user_id: int,
##       k: int = 10,
##       user2idx: dict = None,
##       idx2item: dict = None,
##       popular_items: list = None,
##       device: str = 'cpu',
##   ) -> dict:
##       """
##       Return top-K recommendations with graceful cold-start handling.
##
##       Steps:
##       1. Try: recs = recommend(model, user_id, k, user2idx, idx2item, device)
##          On success → return {'items': recs, 'strategy': 'personalized'}
##       2. Except ValueError (unknown user):
##          a. If popular_items is not None:
##               → return {'items': popular_items[:k], 'strategy': 'popularity_fallback'}
##          b. Else:
##               → return {'items': [], 'strategy': 'unknown_user_no_fallback'}
##       """
##       pass  # YOUR CODE HERE
##
## ── Prototype here ───────────────────────────────────────────────────────────
## ============================================================================

def recommend_with_fallback_proto(
    model,
    user_id: int,
    k: int = 10,
    user2idx: dict = None,
    idx2item: dict = None,
    popular_items: list = None,
    device: str = 'cpu',
) -> dict:
    """Prototype — copy to src/inference.py as recommend_with_fallback()."""
    # YOUR CODE HERE
    pass


# ── Test known user ───────────────────────────────────────────────────────────
known_res = recommend_with_fallback_proto(
    model_loaded, user_ids[0], k=5,
    user2idx=user2idx, idx2item=idx2item,
    popular_items=popular_items,
)
# ── Test unknown user ─────────────────────────────────────────────────────────
unknown_res = recommend_with_fallback_proto(
    model_loaded, 99999999, k=5,
    user2idx=user2idx, idx2item=idx2item,
    popular_items=popular_items,
)

if known_res is not None:
    print(f'✅ Known user   → strategy: {known_res.get("strategy")}  | items: {known_res.get("items", [])[:3]}...')
else:
    print('❌ Implement recommend_with_fallback_proto() above!')

if unknown_res is not None:
    print(f'✅ Unknown user → strategy: {unknown_res.get("strategy")} | items: {unknown_res.get("items", [])[:3]}...')
else:
    print('❌ Implement recommend_with_fallback_proto() above!')

---
## 🏭 PART H: PRODUCTION BEST PRACTICES FOR MLES

This section is **conceptual** — read and discuss. You do not need to implement everything
for the deliverable, but you should reference these practices in your final report.

### H.1: Monitoring Your Serving System

Once deployed, track two types of health:

**Operational health** (is the API working?)
- Request latency p50 / p95 / p99
- Error rate (5xx, 4xx)
- Cold-start rate (% of requests that fall back to popularity)

**Model health** (are the recommendations good?)
- Click-through rate (CTR) — do users click recommended items?
- Coverage — what % of the catalog ever gets recommended?
- Score distribution drift — are model outputs shifting over time?

```python
# Minimal logging wrapper — add to src/inference.py
import time, logging

def timed_recommend(model, user_id, k, **kwargs):
    t0     = time.perf_counter()
    result = recommend_with_fallback(model, user_id, k, **kwargs)
    ms     = (time.perf_counter() - t0) * 1000
    logging.info({'user_id': user_id, 'strategy': result['strategy'],
                  'latency_ms': round(ms, 2), 'k': k})
    return result
```

### H.2: A/B Testing Model Versions

Before replacing a model in production, validate it on real traffic:

```
50% users → Model A (current)
50% users → Model B (new NCF / new hyperparameters)

Measure for 1–2 weeks:
  CTR, session length, return rate

Winner → promoted to 100% traffic
```

Implementation: use a random hash of `user_id % 2` to assign cohort.

### H.3: When to Retrain

Models degrade as user tastes shift and new items appear. Common triggers:

| Trigger | Description |
|---------|-------------|
| **Scheduled** | Retrain weekly/nightly with accumulated new data |
| **Drift-based** | Score distribution shifts beyond a threshold |
| **Performance-based** | Offline Precision@K drops below acceptance bar |

**For your report**: describe how you would automate retraining if this system were
deployed in production for 6 months.

In [ ]:
## ============================================================================
## PART H: LATENCY BENCHMARK & MONITORING DEMO
##
## Add the timed_recommend wrapper to src/inference.py for observability.
## Here we benchmark inference latency locally.
## ============================================================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(message)s',
    datefmt='%H:%M:%S',
)

def timed_recommend(model, user_id: int, k: int,
                    user2idx: dict, idx2item: dict,
                    popular_items: list = None,
                    device: str = 'cpu') -> dict:
    """Inference wrapper that logs latency and strategy."""
    t0     = time.perf_counter()
    result = recommend_with_fallback_proto(
        model, user_id, k=k,
        user2idx=user2idx, idx2item=idx2item,
        popular_items=popular_items, device=device,
    ) or {'items': [], 'strategy': 'not_implemented'}
    ms = (time.perf_counter() - t0) * 1000
    logging.info({'user_id': user_id, 'strategy': result.get('strategy'),
                  'latency_ms': round(ms, 3), 'k': k})
    return result


# ── Benchmark: 20 requests against known users ──────────────────────────────
print('🕐 Latency benchmark (20 requests)\n')
latencies = []
sample    = user_ids[:20]

for uid in sample:
    t0 = time.perf_counter()
    timed_recommend(model_loaded, uid, k=10,
                    user2idx=user2idx, idx2item=idx2item,
                    popular_items=popular_items)
    latencies.append((time.perf_counter() - t0) * 1000)

print(f'\n  p50 : {np.percentile(latencies, 50):.1f} ms')
print(f'  p95 : {np.percentile(latencies, 95):.1f} ms')
print(f'  p99 : {np.percentile(latencies, 99):.1f} ms')
print(f'  max : {max(latencies):.1f} ms')
print(f'\n✅ Target: < 100 ms p99 for online serving (CPU). '
      f'GPU reduces this ~10×.')

# ── Latency histogram ───────────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.hist(latencies, bins=15, color='steelblue', edgecolor='white')
plt.axvline(np.percentile(latencies, 95), color='red', linestyle='--',
            label=f'p95 = {np.percentile(latencies, 95):.1f} ms')
plt.title('Inference Latency Distribution (CPU, local)', fontsize=13)
plt.xlabel('Latency (ms)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

---
## 🔭 PART I: FUTURE DIRECTIONS

### I.1: Content-Based Cold Start

When item metadata is available (genres, descriptions, tags), compute **item content
embeddings** and recommend based on explicit user preferences — no interaction history needed.

```python
# User says: "I like Action and Sci-Fi from the 2010s"
# → Embed item genres via TF-IDF / sentence transformer
# → Cosine similarity between user preference vector and item vectors
# → Return top-K most similar items
```

### I.2: Contextual Recommendations

Standard CF asks: *"What should I recommend to user U?"*  
Contextual CF asks: *"What should I recommend to user U **right now**, given context C?"*

| Context signal | Example |
|---------------|---------|
| Time of day | Action at night, documentaries in the morning |
| Device | Short clips on mobile, full films on TV |
| Recent session | Last 3 items watched → session-based model |
| Mood (inferred) | Browsing pattern → upbeat vs. calm content |

**Extension**: Add context features as additional embedding inputs to NeuralCF.

### I.3: Reinforcement Learning for Long-Term Engagement

Standard models optimize for **single-step prediction** (will user rate this item highly?).
RL optimizes for **cumulative user satisfaction** over many interactions:

$$\text{Goal: maximize } \mathbb{E}\left[\sum_{t=0}^{T} \gamma^t R_t\right]$$

Where $R_t$ = reward at step $t$ (click, session length, return visit).

**Practical entry point**: Multi-armed bandit algorithms (Thompson Sampling, UCB) that
balance exploitation (recommend known-good items) with exploration (try new ones).

### I.4: Scale-Out Patterns for Production

| Pattern | Description |
|---------|-------------|
| **Two-stage retrieval** | FAISS/ScaNN ANN search → fast candidate recall, NCF re-ranks top-500 |
| **Feature store** | Pre-computed user/item features served at low latency (Redis, Feast) |
| **Model registry** | MLflow / W&B tracks all runs, artifacts, and promotion history |
| **Online learning** | Incrementally update embeddings as new interactions arrive |
| **LLM-augmented** | Use GPT-style model for explanation + diversity beyond pure CF |

---
## 📋 PART J: FINAL PROJECT REPORT CHECKLIST

Your final report must demonstrate the **complete ML lifecycle**: data → model → deployment.
Each section should include at least one visualization and a brief written discussion.

### Required Report Sections

| # | Section | What to Cover | Sessions |
|---|---------|--------------|----------|
| 1 | **Problem & Data** | Dataset description, EDA, sparsity, rating distribution | 1 |
| 2 | **Baseline** | Popularity recommender — results as lower bound | 1–2 |
| 3 | **Classic CF** | User/Item CF and Matrix Factorization (SVD) with Precision@K, Recall@K | 2 |
| 4 | **Deep Learning (NCF)** | Architecture choices, training curves, hyperparameter decisions | 3 |
| 5 | **Model Comparison** | Table comparing all models on the same test set | 2–3 |
| 6 | **Deployment** | Inference pipeline design, API diagram, demo screenshot | 4 |
| 7 | **Cold Start** | Strategy chosen, implementation details, tradeoff discussion | 4 |
| 8 | **Lessons Learned** | What worked, what failed, what you'd do differently | all |
| 9 | **Future Work** | Contextual rec, RL, ANN retrieval — concrete next steps | 4 |

---

### Code Deliverables Checklist

**`src/inference.py`**
- [ ] `load_model(checkpoint_path, device)` — loads NeuralCF from .pt checkpoint
- [ ] `recommend(model, user_id, k, user2idx, idx2item, device)` — top-K for known users
- [ ] `recommend_with_fallback(..., popular_items)` — graceful cold-start handling

**`app/main.py`**
- [ ] Model loaded once at startup via `lifespan`
- [ ] `GET /` returns `{"status": "ok"}`
- [ ] `GET /model-info` returns architecture metadata
- [ ] `GET /recommend/{user_id}` returns top-K item IDs with `strategy` field
- [ ] Input validation: `k` range check, HTTP 400/503 errors
- [ ] Cold-start fallback integrated (unknown users get popularity list)

**`app/frontend.py`**
- [ ] Streamlit UI with user ID input and k slider
- [ ] Calls API and renders results table
- [ ] Sidebar shows API connection status
- [ ] Handles API errors and unknown-user warnings gracefully

**Working Demo**
- [ ] `uvicorn app.main:app --reload` starts without errors
- [ ] `GET /recommend/{user_id}` responds in < 200 ms (CPU)
- [ ] Unknown users receive a popularity-based response (not a 500 crash)
- [ ] `streamlit run app/frontend.py` renders UI and fetches live recommendations
- [ ] All 6 smoke tests in Part F pass

---

### Congratulations 🎉

You have walked the complete MLE lifecycle:

```
Raw Data → EDA → Baseline → Classic CF → Deep Learning → Inference → API → UI
   S1       S1      S2          S2            S3            S4       S4    S4
```

**Skills demonstrated:**
- Data engineering (ingestion, preprocessing, train/test split)
- Model development (Popularity, MF, SVD, NCF with PyTorch)
- Offline evaluation (Precision@K, Recall@K, RMSE)
- ML engineering (model serialization, vectorized inference)
- API development (FastAPI, REST, input validation, startup patterns)
- Frontend prototyping (Streamlit)
- Production thinking (monitoring, A/B testing, cold start, retraining strategy)